# ExcelTableCNN — training runbook (Kaggle / Colab)

Trains the table detector on the VEnron2 corpus with its public table-range annotations.

**Before running:**
- Enable a GPU (Kaggle: *Settings → Accelerator → GPU*; Colab: *Runtime → Change runtime type → GPU*). CUDA and mixed precision are picked up automatically (`device="auto"`).
- The first run downloads the corpus (~hundreds of MB) and featurizes it — `.xls` files are read natively via xlrd, **no LibreOffice needed**. Featurized tensors are cached, so reruns are fast.

**Where things land:** on Kaggle use `/kaggle/working` (persisted with the notebook output); on Colab mount Drive if you want the feature cache and checkpoints to survive the session.

In [ ]:
# Environment setup — just the package; .xls files are read natively.
# (LibreOffice is only needed for .xlsb corpora: !apt-get install -y -qq libreoffice,
#  then pass use_libreoffice=True to get_train_test.)
%pip install -q git+https://github.com/Flagro/ExcelTableCNN.git

In [ ]:
import logging, os
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

WORK_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
DATA_DIR = os.path.join(WORK_DIR, 'data')
CHECKPOINT_DIR = os.path.join(WORK_DIR, 'checkpoints')

# On Colab, optionally persist across sessions:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/exceltablecnn/data'
# CHECKPOINT_DIR = '/content/drive/MyDrive/exceltablecnn/checkpoints'

In [ ]:
# Download the corpus + annotations and build cached feature tensors.
# Start small (50/20 sheets); raise or set to None (= all) once the run works.
from excel_table_cnn import get_train_test

train_samples, test_samples = get_train_test(
    data_folder_path=DATA_DIR,
    train_size=50,
    testing_size=20,
)
len(train_samples), len(test_samples)

In [ ]:
from excel_table_cnn import (
    NUM_FEATURES, SpreadsheetDataset, TrainConfig, build_model, train_model,
)

train_dataset = SpreadsheetDataset(train_samples)
print(f'{len(train_dataset)} training sheets ({len(train_dataset.skipped)} skipped)')

model = build_model(in_channels=NUM_FEATURES)
config = TrainConfig(
    epochs=20,
    lr=0.005,
    # device="auto" resolves to CUDA when available (else CPU); AMP follows suit.
    checkpoint_dir=CHECKPOINT_DIR,
)
history = train_model(model, train_dataset, config)

In [ ]:
# Watch the loss components: if loss_objectness or loss_classifier sit at ~0
# from the very start, something upstream is broken (see project docs).
import pandas as pd
history_df = pd.DataFrame(history)
history_df[['loss_total', 'loss_objectness', 'loss_rpn_box_reg',
            'loss_classifier', 'loss_box_reg']].rolling(20).mean().plot(figsize=(10, 4));

In [ ]:
# Evaluate with the Error-of-Boundary metric.
from excel_table_cnn import SpreadsheetDataset, evaluate_model, format_report

test_dataset = SpreadsheetDataset(test_samples)
report = evaluate_model(model, test_dataset)  # device="auto"
print(format_report(report))

In [ ]:
# Error analysis: worst sheets first (highest best-EoB per ground-truth table).
worst = sorted(
    report['per_sheet'],
    key=lambda s: max((e for e in s['best_eob_per_gt'] if e is not None), default=999),
    reverse=True,
)
for sheet in worst[:5]:
    print(sheet['file_path'], sheet['sheet_name'])
    print('  ground truth:', sheet['ground_truth'])
    print('  predictions :', [(p['range'], round(p['score'], 2)) for p in sheet['predictions']])
    print('  best EoB    :', sheet['best_eob_per_gt'])

In [ ]:
# Inference on any .xlsx/.xlsm/.xls file with the trained model:
# from excel_table_cnn import detect_tables
# detect_tables('some_file.xls', model=model)

## Scaling up

- Set `train_size=None` to use every annotated training sheet; raise `epochs` accordingly.
- Checkpoints: `checkpoints/last.pt` is refreshed every epoch — reload with `excel_table_cnn.load_checkpoint(path)`.
- If GPU memory is tight, lower the featurization caps: `get_train_test(..., max_rows=1024, max_cols=256)`.